In [14]:
from pathlib import Path
from typing import Any, Dict, List
import uuid
import httpx

from pymilvus import MilvusClient, DataType
from ollama import Client as OllamaClient

# ---- Config ----
CHUNKING_API_URL = "http://localhost:8002"
MILVUS_URI = "http://localhost:19530"   # Milvus standalone
COLL = "finetuning"

OLLAMA_HOST = "http://localhost:11434"
OLLAMA_EMB_MODEL = "qwen3-embedding:0.6b"

MAX_TEXT = 60000          # max text stored in Milvus text field is 65535
MAX_EMB_CHARS = 8000      # important: keep this low to avoid slow / memory issues
BATCH_SIZE = 256

ollama = OllamaClient(host=OLLAMA_HOST)
client = MilvusClient(uri=MILVUS_URI)


# Embedding

In [ ]:
# def emb_text(text: str) -> List[float]:
#     text = (text or "")[:MAX_EMB_CHARS]
#     out = ollama.embeddings(model=OLLAMA_EMB_MODEL, prompt=text)
#     vec = out.get("embedding")
#     if not isinstance(vec, list) or not vec:
#         raise RuntimeError("Embedding Ollama invalide (liste vide / mauvais format).")
#     return vec


ollama_client = OllamaClient(host="http://localhost:11434")

def emb_text(text: str) -> list[float]:
    return ollama_client.embeddings(model="qwen3-embedding:0.6b", prompt=text)["embedding"]

EMBED_DIM = len(emb_text("ping"))
EMBED_DIM



1024

# Chunking

In [10]:
def chunk_pdf_via_api(
    pdf_path: str | Path,
    *,
    pdf_strategy: str = "structure",
    chunk_size_tokens: int = 300,
    min_chunk_chars: int = 200,
    artifacts_path: str | None = None,
    image_model: str | None = None,
    text_model: str | None = None,
) -> List[Dict[str, Any]]:
    pdf_path = Path(pdf_path).expanduser().resolve()
    params: Dict[str, Any] = {
        "pdf_strategy": pdf_strategy,
        "chunk_size_tokens": chunk_size_tokens,
        "min_chunk_chars": min_chunk_chars,
    }
    if artifacts_path:
        params["artifacts_path"] = artifacts_path
    if image_model:
        params["image_model"] = image_model
    if text_model:
        params["text_model"] = text_model

    with pdf_path.open("rb") as f, httpx.Client(timeout=600.0) as h:
        r = h.post(
            f"{CHUNKING_API_URL}/chunks",
            params=params,
            files={"file": (pdf_path.name, f, "application/pdf")},
        )
        r.raise_for_status()
        chunks = r.json()

    return chunks


In [34]:
PDF_PATH = "pdfs/2310.08560v2.pdf"
ARTIFACTS_PATH = str(Path.home() / ".cache/docling/models")

chunks = chunk_pdf_via_api(
    PDF_PATH,
    pdf_strategy="structure",
    chunk_size_tokens=300,
    min_chunk_chars=200,
    artifacts_path=ARTIFACTS_PATH,
)
len(chunks), chunks[0].keys()


(53,
 dict_keys(['id', 'source', 'text', 'page_no', 'section_title', 'section_path', 'meta']))

# Insert VDB

In [35]:
import uuid
import json

MAX_TEXT = 60000
BATCH_SIZE = 256
COLL = "finetuning"


def insert_new_chunks(items, coll=COLL, default_source="noise", chunk_type="noise"):
    batch = []
    inserted = 0

    for obj in items:
        text = (obj.get("text") or "")[:MAX_TEXT]
        if not text.strip():
            continue

        vec = emb_text(text)  # -> dim 1024 attendu

        batch.append({
            "id": obj.get("id"),   
            "vector": vec,
            "text": text,                             # déclenche BM25 -> sparse via Function
            "source": obj.get("source", default_source),
            "section_path": obj.get("section_path", ""),
            "section_title": obj.get("section_title", ""),
            "page_no": int(obj.get("page_no", -1)) if obj.get("page_no") is not None else -1,
            "chunk_type": obj.get("chunk_type", chunk_type),
        })

        if len(batch) >= BATCH_SIZE:
            client.insert(coll, batch)
            inserted += len(batch)
            batch.clear()

    if batch:
        client.insert(coll, batch)
        inserted += len(batch)

    client.flush(coll)       
    # client.load_collection(coll)  # en général pas nécessaire si déjà load

    return inserted

# with open(JSON_PATH, "r", encoding="utf-8") as f:
#     noise_items = json.load(f)


n = insert_new_chunks(chunks, coll=COLL)
print("Inserted noise chunks:", n)

Inserted noise chunks: 53


In [ ]:
# import uuid
# from typing import Any, Dict, List
# from pymilvus import MilvusClient
# from ollama import Client as OllamaClient

# MILVUS_URI = "http://localhost:19530"
# COLL = "finetuning"   # ta collection

# MAX_TEXT_STORE = 65535
# MAX_TEXT_EMB = 8000
# BATCH_SIZE = 256

# ollama_client = OllamaClient(host="http://localhost:11434")

# def emb_text(text: str) -> list[float]:
#     text = (text or "")[:MAX_TEXT_EMB]
#     return ollama_client.embeddings(model="qwen3-embedding:0.6b", prompt=text)["embedding"]

# def make_noise_id(prefix="chunk"):
#     return f"{prefix}_{uuid.uuid4().hex}"

# def safe_id(x: str, max_len=512) -> str:
#     x = (x or "").strip()
#     return x if len(x) <= max_len else x[:max_len]

# client = MilvusClient(uri=MILVUS_URI)

# def insert_new_chunks(
#     items: List[Dict[str, Any]],
#     *,
#     coll: str,
#     default_source: str,
#     default_chunk_type: str = "doc_chunk",
#     batch_size: int = BATCH_SIZE,
# ) -> int:
#     batch = []
#     inserted = 0

#     for obj in items:
#         text_full = (obj.get("text") or "")[:MAX_TEXT_STORE]
#         if not text_full.strip():
#             continue

#         vec = emb_text(text_full)  # embedding sur texte tronqué (MAX_TEXT_EMB)

#         batch.append({
#             "id": safe_id(obj.get("id") or make_noise_id()),
#             "vector": vec,
#             "text": text_full,  # déclenche BM25->sparse via Function
#             "source": obj.get("source") or default_source,
#             "section_path": obj.get("section_path", "") or "",
#             "section_title": obj.get("section_title", "") or "",
#             "page_no": int(obj.get("page_no", -1)) if obj.get("page_no") is not None else -1,
#             "chunk_type": (obj.get("chunk_type") or "").strip() or default_chunk_type,
#         })

#         if len(batch) >= batch_size:
#             client.insert(collection_name=coll, data=batch)
#             inserted += len(batch)
#             batch.clear()

#     if batch:
#         client.insert(collection_name=coll, data=batch)
#         inserted += len(batch)

#     client.flush(collection_name=coll)
#     return inserted


In [ ]:
# source = Path(PDF_PATH).name
# n = insert_new_chunks(chunks, coll=COLL, default_source=source, default_chunk_type="doc_chunk")
# print("Inserted chunks:", n)


KeyboardInterrupt: 